# Cycle 1 - Modèle de régression linéaire

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from sklearn.pipeline import Pipeline
from agritech_feature_engineering import AgritechFeatureEngineering
from sklearn.model_selection import cross_validate, TimeSeriesSplit
import mlflow
import notebooks.utils as utils
from datetime import datetime as dt

In [3]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [4]:
dataset = pd.read_csv('../data/rafined/dataset.csv', sep=',', na_values=[''], quotechar='"')

In [5]:
dataset.drop(columns=['Area ISO Code', 'Area ISO Alpha-2 Code', 'Area ISO Alpha-3 Code'], inplace=True)

In [6]:
X_train, X_test, y_train, y_test = utils.test_train_time_split(dataset)

Année de césure calculée : 2009
Proportion réelle train : 78.73% (20349 lignes)
Proportion réelle test  : 21.27% (5499 lignes)


In [7]:
X_train = X_train.sort_values('Year')
y_train = y_train.loc[X_train.index]

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_features = [
    'avg_temp',
    'average_rain_fall_mm_per_year',
    'pesticides_tonnes',
    'pesticides_kg_ha',
    'aridity',
    'avg_temp_square',
    'missing_pesticide_value',
    'missing_rain_value',
    'missing_temp_value'
]

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('area_encoding', TargetEncoder(target_type='continuous', random_state=42), ['Area']),
        ('item_encoding', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Item']),
        ('num_pipeline', numeric_transformer, numeric_features),
    ],
    remainder='drop'
)

In [9]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

In [10]:
pipeline = Pipeline([
    ('features', AgritechFeatureEngineering()),
    ('preprocessor', preprocessor),
    ('model', model)
])

In [11]:
mlflow.set_experiment("Agriculture_optimization")
mlflow.end_run()
mlflow.set_system_metrics_sampling_interval(1)

In [12]:
scoring = {'r2': 'r2', 'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'}

with mlflow.start_run(run_name="Cycle 1 - Régression linéaire", log_system_metrics=True):

    mlflow.set_tags({
        'date': dt.now().strftime("%Y-%m-%d"),
        'cycle': '1_regression',
        'model_type': 'linear_regression',
        'mlflow.note.content': (
            "Régression linéaire : modèle de base pour prédire le rendement agricole "
            "en utilisant les variables climatiques et agricoles."
        ),
    })

    mlflow.log_params({
        'split': 'temporel',
        'split_year': int(X_test['Year'].min()),
        'model': 'LinearRegression',
        'cv': 'TimeSeriesSplit 5 plis',
        'target_transform': 'aucune',
        'features': 'Area, Item, avg_temp, rain, pesticides, aridity, avg_temp_square, missing_*',
    })

    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=TimeSeriesSplit(5),
        scoring=scoring,
        return_train_score=True,
        error_score='raise'
    )

    mlflow.log_metrics({
        'r2': cv_results['test_r2'].mean(),
        'mae': -cv_results['test_mae'].mean(),
        'rmse': -cv_results['test_rmse'].mean(),
        'train_r2': cv_results['train_r2'].mean(),
        'train_mae': -cv_results['train_mae'].mean(),
        'train_rmse': -cv_results['train_rmse'].mean(),
        'fit_time_mean': cv_results['fit_time'].mean()
    })

    print('--- Validation Fold Results ---')
    print(f"Validation R2 : {cv_results['test_r2']}, R2 moyen : {cv_results['test_r2'].mean()}")
    print(f"Validation MAE : {cv_results['test_mae']}, MAE moyen : {cv_results['test_mae'].mean()}")
    print(f"Validation RMSE : {cv_results['test_rmse']}, RMSE moyen : {cv_results['test_rmse'].mean()}")

    print('--- Train Fold Results ---')
    print(f"Train R2 moyen : {cv_results['train_r2'].mean()}")
    print(f"Train MAE moyen : {-cv_results['train_mae'].mean()}")
    print(f"Train RMSE moyen : {-cv_results['train_rmse'].mean()}")

2026/11/19 14:21:43 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/19 14:21:43 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
/Users/auncly/Documents/Formation data scientist/12 - Concevez un système de recommandations pour une agriculture optimisée par les données/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEnco

--- Validation Fold Results ---
Validation R2 : [0.64692095 0.66998133 0.64156999 0.6739872  0.67176625], R2 moyen : 0.660845141697359
Validation MAE : [-26605.11505831 -26785.88365797 -29365.81062054 -29076.14203107
 -29951.49809028], MAE moyen : -28356.88989163386
Validation RMSE : [-41028.65329493 -39873.6734919  -44656.39265205 -43018.374661
 -44279.74191002], RMSE moyen : -42571.36720198146
--- Train Fold Results ---
Train R2 moyen : 0.6785094781721638
Train MAE moyen : 26304.608512786275
Train RMSE moyen : 38744.79820949516
🏃 View run Cycle 1 - Régression linéaire at: http://127.0.0.1:5000/#/experiments/1/runs/7deb60d89f5b408a98021c604cee81cf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Conclusion du cycle 1 : régression linéaire

Premier modèle de machine learning, une régression linéaire. Elle utilise la culture, le pays, la pluie, la température, les pesticides et deux variables construites (indice d'aridité, température au carré).

**Validation** : validation croisée temporelle (`TimeSeriesSplit`, 5 plis) sur les données d'entraînement (avant 2009), triées par année. À chaque pli, le modèle apprend sur les années passées et il est évalué sur les années qui suivent, comme en conditions réelles.

**Résultats** :

| Métrique | Validation (moyenne) | Validation (plis) | Train (moyenne) |
|---|---|---|---|
| R² | 0.660 | de 0.641 à 0.674 | 0.678 |
| MAE | 28 350 hg/ha | de 26 622 à 29 943 | 26 290 hg/ha |
| RMSE | 42 594 hg/ha | de 39 886 à 44 670 | 38 764 hg/ha |

Rappel de la baseline naïve : R² = 0.836, RMSE = 32 646 hg/ha, MAE = 16 461 hg/ha.

- La régression linéaire fait nettement moins bien que la baseline naïve : elle explique 66 % de la variance du rendement contre 84 %, et se trompe en moyenne d'environ 2,8 tonnes par hectare, contre 1,6 tonne. Les deux évaluations ne portent pas exactement sur les mêmes années, mais l'écart est trop grand pour venir de cette différence. Ce modèle n'est pas utilisable pour conseiller un agriculteur.
- **Pas de surapprentissage** : les scores sur le train (R² 0.678) et en validation (R² 0.660) sont très proches, et les plis sont stables (R² entre 0.64 et 0.67). Le problème est l'inverse, un **sous-apprentissage** : le modèle est trop simple pour représenter les données, même celles qu'il a vues à l'entraînement.
- La RMSE (42 594) vaut environ 1,5 fois la MAE (28 350) : quelques très grosses erreurs pèsent lourd.

**Pistes pour les cycles suivants** :
- Transformer la cible avec un logarithme (`log(1 + rendement)`) : la distribution très asymétrique devient plus régulière, donc plus faciles à apprendre pour un modèle linéaire.
- Passer à d'autres modèles (Random Forest, Gradient Boosting).
